In [1]:
## %% [markdown]
# # 03. PCA композиционного пространства и пул незаполненных областей
# Проект: Fe-Ni-Co-Al-Cr-Ti alloy composition space

# %%
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from src.alloy_space.descriptors import calculate_descriptors

FIGURES_DIR = BASE_DIR / "outputs" / "figures"
TABLES_DIR = BASE_DIR / "outputs" / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="ticks", font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

# %%
# 1. Загрузка данных
df_comp = pd.read_parquet(BASE_DIR / "data" / "processed" / "dataset_compositions_unique.parquet")
BASE_6 = ['Ni', 'Co', 'Fe', 'Cr', 'Al', 'Ti']
assert set(BASE_6) <= set(df_comp.columns), "Нет колонок базовых элементов"

N_ALLOYS = len(df_comp)
X_base6 = df_comp[BASE_6].values


# Al+Ti и Al/Ti для литературных составов (для рис. 08 и текста про окно)
df_comp['Al_plus_Ti'] = df_comp['Al'] + df_comp['Ti']
df_comp['Al_to_Ti_ratio'] = df_comp['Al'] / df_comp['Ti'].replace(0, np.nan)

cols = ['Al_plus_Ti', 'Al_to_Ti_ratio']
print(f"Литературных составов: {N_ALLOYS}")
print("Все составы:")
print(df_comp[cols].describe(percentiles=[.05, .5, .95]).round(2).to_string())
print("\nСоставы с Target state = 1:")
print(df_comp.loc[df_comp['Target state'] == 1, cols]
      .describe(percentiles=[.05, .5, .95]).round(2).to_string())

Литературных составов: 161
Все составы:
       Al_plus_Ti  Al_to_Ti_ratio
count      161.00          116.00
mean        10.71            1.67
std          5.63            1.27
min          0.00            0.00
5%           4.00            0.23
50%          9.10            1.35
95%         17.30            3.29
max         40.00            8.00

Составы с Target state = 1:
       Al_plus_Ti  Al_to_Ti_ratio
count       87.00           77.00
mean        10.46            1.67
std          3.75            1.33
min          2.70            0.00
5%           5.30            0.50
50%          9.10            1.30
95%         16.10            3.03
max         18.10            8.00


In [2]:
# %%
# 2. PCA на стандартизованных концентрациях базовых элементов
scaler_base6 = StandardScaler()
X_base6_scaled = scaler_base6.fit_transform(X_base6)

pca = PCA(n_components=len(BASE_6))
X_pca_full = pca.fit_transform(X_base6_scaled)

for i in range(3):
    df_comp[f'PC{i + 1}'] = X_pca_full[:, i]

exp_var = pca.explained_variance_ratio_ * 100
variance_table = pd.DataFrame({
    'Component': [f'PC{i + 1}' for i in range(len(exp_var))],
    'Explained variance, %': exp_var.round(2),
    'Cumulative, %': np.cumsum(exp_var).round(2)})
print(variance_table.to_string(index=False))

loadings = pd.DataFrame(pca.components_[:3].T, columns=['PC1', 'PC2', 'PC3'], index=BASE_6)
print("\n=== Нагрузки элементов на PC1–PC3 ===")
print(loadings.round(3))
print("\n=== Вклад элемента в дисперсию оси, % ===")
print(((loadings ** 2) * 100).round(1))

with pd.ExcelWriter(TABLES_DIR / "pca_summary.xlsx", engine="openpyxl") as writer:
    variance_table.to_excel(writer, sheet_name="Explained_variance", index=False)
    loadings.round(4).rename_axis("Element").reset_index().to_excel(
        writer, sheet_name="Loadings_PC1-3", index=False)

Component  Explained variance, %  Cumulative, %
      PC1                  28.89          28.89
      PC2                  24.10          52.99
      PC3                  18.01          71.00
      PC4                  15.94          86.94
      PC5                  12.17          99.12
      PC6                   0.88         100.00

=== Нагрузки элементов на PC1–PC3 ===
      PC1    PC2    PC3
Ni  0.667 -0.111  0.020
Co -0.500 -0.569 -0.180
Fe -0.229  0.658 -0.304
Cr -0.127  0.126  0.927
Al  0.301  0.299 -0.117
Ti  0.381 -0.354 -0.052

=== Вклад элемента в дисперсию оси, % ===
     PC1   PC2   PC3
Ni  44.5   1.2   0.0
Co  25.0  32.4   3.2
Fe   5.2  43.2   9.2
Cr   1.6   1.6  85.9
Al   9.1   9.0   1.4
Ti  14.5  12.6   0.3


In [3]:
# %%
# 3. 3D-сетка, возврат в состав, L1_2-окно, пул
MARGIN, GRID_N = 0.15, 35
pcs = df_comp[['PC1', 'PC2', 'PC3']].values

axes_1d = [np.linspace(pcs[:, k].min() - MARGIN * np.ptp(pcs[:, k]),
                       pcs[:, k].max() + MARGIN * np.ptp(pcs[:, k]), GRID_N) for k in range(3)]
grid = np.column_stack([g.ravel() for g in np.meshgrid(*axes_1d)])

nn = NearestNeighbors(n_neighbors=1).fit(pcs)
df_grid = pd.DataFrame(grid, columns=['PC1', 'PC2', 'PC3'])
df_grid['dist_to_nearest_alloy'] = nn.kneighbors(grid)[0].ravel()

full = np.zeros((len(grid), pca.n_components_))
full[:, :3] = grid
comp = pd.DataFrame(scaler_base6.inverse_transform(pca.inverse_transform(full)), columns=BASE_6)
df_grid = pd.concat([df_grid, comp], axis=1)

df_valid = df_grid[(df_grid[BASE_6] >= -1.0).all(axis=1)].copy()
df_valid[BASE_6] = df_valid[BASE_6].clip(lower=0)
df_valid[BASE_6] = df_valid[BASE_6].div(df_valid[BASE_6].sum(axis=1), axis=0) * 100.0
df_valid = pd.concat([df_valid, calculate_descriptors(df_valid)], axis=1)

def in_l12_window(d):
    return ((d['VEC'] >= 7.85)
            & d['Al_plus_Ti'].between(8.0, 12.5)
            & d['Al_to_Ti_ratio'].between(0.9, 2.2)
            & (d['delta_atomic_radius'] <= 6.0))

valid_l12 = df_valid[in_l12_window(df_valid)].copy()
gap_threshold = float(np.percentile(valid_l12['dist_to_nearest_alloy'], 85))
gap_pool = valid_l12[valid_l12['dist_to_nearest_alloy'] >= gap_threshold].copy()

print(f"Узлов сетки: {len(df_grid)}; физически валидных: {len(df_valid)}; "
      f"в L1_2-окне: {len(valid_l12)}; порог 85-го перцентиля: {gap_threshold:.3f}; пул: {len(gap_pool)}")
pool_cols = BASE_6 + ['VEC', 'Al_plus_Ti', 'Al_to_Ti_ratio', 'delta_atomic_radius', 'dist_to_nearest_alloy']
gap_pool[pool_cols].round(2).to_excel(TABLES_DIR / "pca3d_gap_pool_base6.xlsx", index=False)

Узлов сетки: 42875; физически валидных: 12798; в L1_2-окне: 887; порог 85-го перцентиля: 0.907; пул: 133


In [4]:
# %%
# 4. Кандидаты (выбраны вручную по интерактивному 3D-графику) и выплавленные образцы
CANDIDATES = pd.DataFrame([
    [38.81, 28.87,  5.04, 15.25, 5.80, 6.22],   # 1
    [33.65, 25.21,  4.62, 27.22, 4.80, 4.49],   # 2
    [37.20, 19.68,  1.20, 31.94, 5.10, 4.89],   # 3
    [40.61, 14.79,  5.84, 26.59, 7.02, 5.15],   # 4
    [28.37, 22.78, 13.12, 27.48, 5.57, 2.68],   # 5
    [32.87, 36.51, 16.60,  2.17, 6.59, 5.26],   # 6
    [26.35, 40.65, 20.19,  3.12, 5.76, 3.93],   # 7
    [26.96, 42.35, 14.60,  7.05, 4.65, 4.39],   # 8
    [25.49, 48.32, 16.07,  1.14, 4.36, 4.62],   # 9
    [22.27, 45.77, 22.79,  0.59, 5.20, 3.37],   # 10
    [41.81, 20.62,  2.04, 23.16, 6.20, 6.17],   # 11
    [41.34, 18.65,  0.00, 28.39, 5.71, 5.91],   # 12
], columns=BASE_6)
CANDIDATES.insert(0, 'Alloy_Number', range(1, len(CANDIDATES) + 1))
CANDIDATES['Cluster'] = ['Upper'] * 5 + ['Lower'] * 5 + ['Upper'] * 2   # 11-12: проверьте

def project(comp_df):
    X = comp_df[BASE_6].values.astype(float)
    z = pca.transform(scaler_base6.transform(X))
    out = comp_df.copy()
    out[['PC1', 'PC2', 'PC3']] = z[:, :3]
    return out

cand = project(CANDIDATES)
cand = pd.concat([cand, calculate_descriptors(cand)], axis=1)
cand['in_L12_window'] = in_l12_window(cand)
cand['dist_to_nearest_alloy'] = nn.kneighbors(cand[['PC1', 'PC2', 'PC3']].values)[0].ravel()
cand['beyond_gap_threshold'] = cand['dist_to_nearest_alloy'] >= gap_threshold
print("Суммы составов кандидатов:", CANDIDATES[BASE_6].sum(axis=1).round(2).tolist())
print(cand[['Alloy_Number', 'Cluster', 'Al_plus_Ti', 'Al_to_Ti_ratio', 'VEC',
            'in_L12_window', 'dist_to_nearest_alloy', 'beyond_gap_threshold']].round(2).to_string(index=False))
cand.round(3).to_excel(TABLES_DIR / "candidates_12.xlsx", index=False)

# Экспериментальные составы плавок по первоначальным кандидатам № 1–12
EXPERIMENTAL = pd.DataFrame([
    [38.19, 29.31,  5.28, 15.51, 5.39, 6.33],   # 1
    [32.70, 25.30,  4.76, 27.65, 4.99, 4.62],   # 2
    [34.41, 23.90,  1.33, 30.41, 5.05, 4.91],   # 3
    [40.06, 14.91,  6.11, 27.09, 6.71, 5.15],   # 4
    [27.74, 22.80, 13.18, 27.77, 5.87, 2.64],   # 5
    [32.07, 36.64, 17.07,  2.29, 6.64, 5.31],   # 6
    [25.81, 40.68, 20.64,  3.20, 5.79, 3.89],   # 7
    [26.21, 42.24, 14.95,  7.36, 4.85, 4.41],   # 8
    [24.64, 48.37, 16.57,  1.23, 4.54, 4.67],   # 9
    [21.80, 45.59, 23.37,  0.56, 5.31, 3.38],   # 10
    [40.54, 20.53,  2.14, 23.41, 7.09, 6.31],   # 11
    [40.55, 18.82,  0.00, 29.08, 5.56, 6.00],   # 12
], columns=BASE_6)

EXPERIMENTAL.insert(0, 'Alloy_Number', range(1, len(EXPERIMENTAL) + 1))
EXPERIMENTAL['Cluster'] = ['Upper'] * 5 + ['Lower'] * 5 + ['Upper'] * 2

experimental = project(EXPERIMENTAL)
experimental['dist_to_nearest_alloy'] = nn.kneighbors(
    experimental[['PC1', 'PC2', 'PC3']].values
)[0].ravel()

print(
    "Суммы экспериментальных составов:",
    EXPERIMENTAL[BASE_6].sum(axis=1).round(4).tolist()
)

Суммы составов кандидатов: [99.99, 99.99, 100.01, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 99.99, 100.0, 100.0]
 Alloy_Number Cluster  Al_plus_Ti  Al_to_Ti_ratio  VEC  in_L12_window  dist_to_nearest_alloy  beyond_gap_threshold
            1   Upper       12.02            0.93 8.22           True                   1.03                  True
            2   Upper        9.29            1.07 7.96           True                   0.94                  True
            3   Upper        9.99            1.04 7.85           True                   1.27                  True
            4   Upper       12.17            1.36 7.87           True                   0.92                  True
            5   Upper        8.25            2.08 7.86           True                   0.93                  True
            6   Lower       11.85            1.25 8.44           True                   0.87                 False
            7   Lower        9.69            1.47 8.43           True             

In [5]:
# %%
# 5. Итоговый график
import plotly.graph_objects as go

LABELS = {
    'lit': 'Literature alloys',
    'pool': 'Gap pool (hidden by default)',
    'cand': 'Candidate alloys',
    'exp': 'Experimental alloys',
}
CAMERA = dict(eye=dict(x=1.6, y=1.6, z=0.9))

def fmt(v):
    return f"{v:.2f}" if isinstance(v, (float, np.floating)) else str(v)

def hover_text(df, prefix=None, extra=()):
    out = []
    for i, (_, r) in enumerate(df.iterrows()):
        head = f"<b>{prefix[i]}</b><br>" if prefix is not None else ""
        comps = ", ".join(f"{c}={r[c]:.1f}" for c in BASE_6)
        tail = "".join(f"<br>{c}={fmt(r[c])}" for c in extra)
        out.append(head + comps + tail)
    return out

# %%
# Связи между кандидатами верхнего кластера (бывшие жёлтые + бывшие бирюзовые)
UPPER_LINKS = [
    (5, 4), (5, 2), (5, 3), (3, 4), (3, 2), (1, 4), (1, 2),            # были жёлтыми
    (12, 2), (12, 3), (12, 4), (12, 11), (11, 1), (11, 2), (11, 4),    # были бирюзовыми
]
LOWER_LINKS = [(10, 7), (10, 9), (10, 8), (8, 7), (8, 9), (6, 8), (6, 9), (6, 7)]   # нижний кластер настроим позже
LINKS = sorted({tuple(sorted(p)) for p in UPPER_LINKS + LOWER_LINKS})
LINK_COLOR = 'goldenrod'

points_by_number = {int(r.Alloy_Number): (r.PC1, r.PC2, r.PC3) for r in cand.itertuples()}
assert all(a in points_by_number and b in points_by_number for a, b in LINKS), "Связь на несуществующий номер"

def links_trace(pairs, color=LINK_COLOR, name='Links between candidates', width=4):
    xs, ys, zs = [], [], []
    for a, b in pairs:
        pa, pb = points_by_number[a], points_by_number[b]
        xs += [pa[0], pb[0], None]
        ys += [pa[1], pb[1], None]
        zs += [pa[2], pb[2], None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                        line=dict(color=color, width=width),
                        name=name, hoverinfo='skip')

# %%
# Объёмная оболочка вокруг узлов gap_pool

# Главный параметр ширины области:
# 1.00 — базовая оболочка, связанная с шагом исходной PCA-сетки
# 1.10 — радиус окрестностей увеличен на 10%
# 1.25 — радиус окрестностей увеличен на 25%
GAP_ENVELOPE_SCALE = 1.10

# Детализация поверхности: больше — плавнее, но тяжелее HTML
GAP_SURFACE_RESOLUTION = 70

# Прозрачность оболочки
GAP_SURFACE_OPACITY = 0.18
GAP_SURFACE_COLOR = 'mediumpurple'


def build_gap_envelope():
    if gap_pool.empty:
        raise ValueError("gap_pool пуст: невозможно построить оболочку.")

    if GAP_ENVELOPE_SCALE <= 0:
        raise ValueError("GAP_ENVELOPE_SCALE должен быть больше нуля.")

    pool_xyz = gap_pool[['PC1', 'PC2', 'PC3']].to_numpy(dtype=float)

    # Шаг исходной сетки вдоль каждой оси PCA
    grid_steps = np.array([
        np.abs(axis[1] - axis[0])
        for axis in axes_1d
    ])

    # Базовый радиус — половина пространственной диагонали ячейки.
    # Это визуальная окрестность узла, не оценка погрешности состава.
    base_radius = 0.5 * np.linalg.norm(grid_steps)
    radius = base_radius * GAP_ENVELOPE_SCALE

    # Дополнительное пространство, чтобы поверхность была замкнутой
    padding = 1.5 * radius
    lower = pool_xyz.min(axis=0) - padding
    upper = pool_xyz.max(axis=0) + padding

    surface_axes = [
        np.linspace(lower[k], upper[k], GAP_SURFACE_RESOLUTION)
        for k in range(3)
    ]

    gx, gy, gz = np.meshgrid(*surface_axes, indexing='ij')
    surface_xyz = np.column_stack([
        gx.ravel(),
        gy.ravel(),
        gz.ravel()
    ])

    # Расстояние до ближайшего узла gap_pool
    pool_nn = NearestNeighbors(n_neighbors=1).fit(pool_xyz)
    distances = pool_nn.kneighbors(
        surface_xyz,
        return_distance=True
    )[0].ravel()

    return surface_xyz, distances, radius


gap_surface_xyz, gap_surface_distances, gap_envelope_radius = (
    build_gap_envelope()
)

print(
    f"Gap envelope: scale={GAP_ENVELOPE_SCALE:.2f}, "
    f"radius={gap_envelope_radius:.3f} PCA units"
)

def make_figure(show_pool=False, show_window=False, links=None,
                width=1000, height=750):
    fig = go.Figure()

    # Литературные составы
    fig.add_trace(go.Scatter3d(
        x=df_comp['PC1'],
        y=df_comp['PC2'],
        z=df_comp['PC3'],
        mode='markers',
        marker=dict(
            size=5,
            color='steelblue',
            opacity=0.7,
            line=dict(width=0.5, color='black')
        ),
        name=f"{LABELS['lit']} (n={len(df_comp)})",
        text=hover_text(df_comp),
        hoverinfo='text'
    ))

    # Все узлы композиционного окна
    if show_window:
        fig.add_trace(go.Scatter3d(
            x=valid_l12['PC1'],
            y=valid_l12['PC2'],
            z=valid_l12['PC3'],
            mode='markers',
            marker=dict(
                size=3,
                color=valid_l12['dist_to_nearest_alloy'],
                colorscale='Viridis',
                cmin=0,
                cmax=valid_l12['dist_to_nearest_alloy'].max(),
                opacity=0.5,
                showscale=True,
                colorbar=dict(
                    title='dist to nearest<br>literature alloy',
                    len=0.6
                )
            ),
            name=f"L1₂-window nodes (n={len(valid_l12)})",
            text=hover_text(
                valid_l12,
                extra=[
                    'Al_plus_Ti',
                    'Al_to_Ti_ratio',
                    'VEC',
                    'delta_atomic_radius',
                    'dist_to_nearest_alloy'
                ]
            ),
            hoverinfo='text'
        ))

    # Пул разреженных областей
        # Полупрозрачные области вместо отдельных узлов gap_pool
    if show_pool:
        fig.add_trace(go.Isosurface(
            x=gap_surface_xyz[:, 0],
            y=gap_surface_xyz[:, 1],
            z=gap_surface_xyz[:, 2],
            value=gap_surface_distances,
            isomin=gap_envelope_radius,
            isomax=gap_envelope_radius,
            surface_count=1,
            colorscale=[
                [0.0, GAP_SURFACE_COLOR],
                [1.0, GAP_SURFACE_COLOR]
            ],
            opacity=GAP_SURFACE_OPACITY,
            showscale=False,
            showlegend=True,
            caps=dict(
                x_show=False,
                y_show=False,
                z_show=False
            ),
            name=(
                "Gap-pool envelope "
                f"(radius scale={GAP_ENVELOPE_SCALE:.2f})"
            ),
            hoverinfo='skip'
        ))

    # Расчётные кандидаты № 1–12
    fig.add_trace(go.Scatter3d(
        x=cand['PC1'],
        y=cand['PC2'],
        z=cand['PC3'],
        mode='markers+text',
        marker=dict(
            size=9,
            color='gold',
            symbol='diamond',
            line=dict(width=2, color='black')
        ),
        text=cand['Alloy_Number'].astype(str),
        textposition='top center',
        textfont=dict(size=13, color='black'),
        hovertext=hover_text(
            cand,
            prefix=[
                f"Candidate {n}"
                for n in cand['Alloy_Number']
            ],
            extra=['Cluster', 'dist_to_nearest_alloy']
        ),
        hoverinfo='text',
        name=f"{LABELS['cand']} (n={len(cand)})"
    ))

    # Экспериментальные составы № 1–12
    fig.add_trace(go.Scatter3d(
        x=experimental['PC1'],
        y=experimental['PC2'],
        z=experimental['PC3'],
        mode='markers+text',
        marker=dict(
            size=9,
            color='darkorange',
            symbol='circle',
            line=dict(width=2, color='black')
        ),
        text=[
            f"E{n}"
            for n in experimental['Alloy_Number']
        ],
        textposition='bottom center',
        textfont=dict(size=13, color='darkorange'),
        hovertext=hover_text(
            experimental,
            prefix=[
                f"Experimental alloy {n}"
                for n in experimental['Alloy_Number']
            ],
            extra=['Cluster', 'dist_to_nearest_alloy']
        ),
        hoverinfo='text',
        name=f"{LABELS['exp']} (n={len(experimental)})"
    ))


    if links:
        fig.add_trace(links_trace(links))

    fig.update_layout(
        template='plotly_white',
        width=width,
        height=height,
        legend=dict(itemsizing='constant'),
        scene=dict(
            xaxis_title=f"PC1 ({exp_var[0]:.1f}%)",
            yaxis_title=f"PC2 ({exp_var[1]:.1f}%)",
            zaxis_title=f"PC3 ({exp_var[2]:.1f}%)",
            camera=CAMERA
        )
    )

    return fig

Gap envelope: scale=1.10, radius=0.235 PCA units


In [6]:
make_figure(
    show_pool=True,
    show_window=False,
    links=LINKS
).write_html(
    str(FIGURES_DIR / "pca3d_composition_space_regions.html")
)

fig_static = make_figure(
    show_pool=True,
    show_window=False,
    links=LINKS
)
fig_static.show()